In [ ]:
# 데이터 불러오고, 이미지 크기 맞추기, 텐서 맞추기, 모델 쪼개서 넘겨주는 과정
# 모델 로드
# 모델 수정
# 손실함수 옵티마이저 설정
# 학습 루프 돌리기(모델 넣고, 예측값 구하고, 손실계산, 역전파)

In [1]:
#데이터 불러오기
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim import lr_scheduler
import numpy as np
import torchvision
from torchvision import datasets, models, transforms
import time
import os
import copy
from torch.utils.data import random_split, Dataset

# 1. 데이터 전처리 설정
data_transforms = {
    'train': transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

data_dir = '/content/drive/MyDrive/AICOSS 2026 WE-Meet/data/processed'

# 전체 데이터를 스캔합니다.
full_dataset = datasets.ImageFolder(data_dir)

# 순서 그대로 강제 지정
# 순서: ['비닐'(0), '스티로폼'(1), '유리병'(2), '종이'(3), '캔'(4), '페트병'(5), '플라스틱'(6)]
target_class_indices = [3, 6]  # 종이(3번), 플라스틱(6번) 인덱스를 직접 지정

# 인덱스 번호에 해당하는 실제 폴더명을 가져옵니다.
target_classes = [full_dataset.classes[i] for i in target_class_indices]

print(f"★ 지정된 클래스 폴더명: {target_classes}")
print(f"★ 지정된 클래스 원본 번호: {target_class_indices}")

# 확보한 정수 번호와 일치하는 이미지 샘플만 안전하게 솎아냅니다.
filtered_samples = [
    sample for sample in full_dataset.samples
    if sample[1] in target_class_indices
]

print(f"★ 최종 확보된 이미지 개수: {len(filtered_samples)}장")

# 원본 클래스 번호(3, 6)를 0번과 1번으로 리매핑하는 딕셔너리 생성
class_mapping = {raw_idx: new_idx for new_idx, raw_idx in enumerate(target_class_indices)}

# 인덱스 버그와 전처리 병목을 차단하는 최종 커스텀 데이터셋 정의
class CustomSelectedDataset(Dataset):
    def __init__(self, samples, class_mapping, transform=None):
        self.samples = samples
        self.class_mapping = class_mapping
        self.transform = transform
        self.loader = full_dataset.loader

    def __getitem__(self, index):
        path, target = self.samples[index]
        sample = self.loader(path)
        if self.transform is not None:
            sample = self.transform(sample)
        # 중요: 모델이 연산 가능하도록 라벨을 0 또는 1로 변환하여 반환
        return sample, self.class_mapping[target]

    def __len__(self):
        return len(self.samples)


# 필터링된 순수 종이/플라스틱 전체 데이터셋 생성
selected_dataset = CustomSelectedDataset(filtered_samples, class_mapping)
class_names = target_classes

# 데이터를 학습용(80%)과 검증용(20%)으로 분할
train_size = int(0.8 * len(selected_dataset))
val_size = len(selected_dataset) - train_size
train_sub, val_sub = random_split(selected_dataset, [train_size, val_size])

#  학습(train)과 검증(val) 서브셋에 각각 올바른 데이터 증강 기법 적용
train_dataset = CustomSelectedDataset([selected_dataset.samples[i] for i in train_sub.indices], class_mapping, transform=data_transforms['train'])
val_dataset = CustomSelectedDataset([selected_dataset.samples[i] for i in val_sub.indices], class_mapping, transform=data_transforms['val'])

# 배치 사이즈를 32로 늘리고 일꾼(worker) 수를 2로 최적화하여 연산 속도 극대화
dataloaders = {
    'train': torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2),
    'val': torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)
}

dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}

# GPU 사용 설정
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")


# 2. 학습용 공통 함수 정의
def train_model(model, criterion, optimizer, scheduler, num_epochs=3):
    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}') #오차 정도, 정확도

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())

        print()

    time_elapsed = time.time() - since
    print(f'Training complete in {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    print(f'Best val Acc: {best_acc:.4f}')

    model.load_state_dict(best_model_wts)
    return model


# 3. ResNet 모델 불러오기 및 수정
model_ft = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
num_ftrs = model_ft.fc.in_features

# len(class_names)는 2가 되므로 출력 노드가 2개인 종이/플라스틱 전용 분류기가 됩니다.
model_ft.fc = nn.Linear(num_ftrs, len(class_names))
model_ft = model_ft.to(device)

# 4. 손실함수 및 최적화 기법 설정
criterion = nn.CrossEntropyLoss()
optimizer_ft = optim.SGD(model_ft.parameters(), lr=0.001, momentum=0.9)
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=7, gamma=0.1)

# 5. 모델 학습 시작
model_ft = train_model(model_ft, criterion, optimizer_ft, exp_lr_scheduler, num_epochs=3)

★ 지정된 클래스 폴더명: ['비닐', '유리병', '종이', '플라스틱']
★ 지정된 클래스 원본 번호: [0, 2, 3, 6]
★ 최종 확보된 이미지 개수: 7641장
Epoch 0/2
----------


KeyboardInterrupt: 